# 개별종목 조합I — RandomForest

`기본모델/02.RandomForest.ipynb`과 같은 `models.random_forest.build_random_forest_baseline`을 가져오고
조합I 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.random_forest import build_random_forest_baseline  # noqa: E402

MODEL_NAME = 'RandomForest'
MODEL_BUILDER = build_random_forest_baseline


In [2]:
# 2. 조합I의 피처 값만 지정합니다.
import json

COMBINATION = 'I'
FEATURE_COLUMNS = (
    'atr_ratio',
    'bb_bandwidth',
    'hv_regime',
    'five_day_return',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "pr_auc_macro_ovr",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "pr_auc_macro_ovr",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합I 피처: ('atr_ratio', 'bb_bandwidth', 'hv_regime', 'five_day_return')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4122,0.5012,-0.0890,0.3553,0.3563,0.0373,0.3511,0.2577,0.3289
1,2,balanced,980,20150123,20150421,0.3637,0.3978,-0.0341,0.3495,0.3508,0.0305,0.3529,0.2765,0.3251
2,3,NaN,1210,20151228,20160328,0.3583,0.3762,-0.0178,0.3536,0.3539,0.0327,0.3538,0.3433,0.3516
3,4,NaN,1439,20161202,20170228,0.3880,0.4617,-0.0737,0.3324,0.3391,0.0128,0.3518,0.2387,0.3069
4,5,balanced,1669,20171113,20180207,0.3713,0.3901,-0.0187,0.3610,0.3618,0.0441,0.3607,0.3072,0.3441
5,6,balanced,1899,20181024,20190118,0.3833,0.3725,0.0108,0.3831,0.3857,0.0780,0.3838,0.4210,0.3950
6,7,balanced,2129,20190930,20191224,0.4160,0.4781,-0.0622,0.3660,0.3701,0.0639,0.3686,0.3035,0.3558
7,8,balanced,2359,20200902,20201130,0.3588,0.3476,0.0112,0.3587,0.3625,0.0432,0.3610,0.4022,0.3722
8,9,balanced,2589,20210806,20211105,0.3639,0.3916,-0.0278,0.3546,0.3604,0.0373,0.3615,0.3113,0.3417
9,10,balanced,2818,20220714,20221012,0.3516,0.3454,0.0062,0.3504,0.3515,0.0273,0.3496,0.3410,0.3476


,OOS 폴드 평균
accuracy,0.3734
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0235
macro_f1,0.3561
balanced_accuracy,0.3584
mcc,0.0395
pr_auc_macro_ovr,0.3600
down_recall,0.3258
core_harmonic_mean,0.3481


재실행 명령: python scripts/run_stock_model_experiment.py
